# Phase 3 Step 1-pre -- CPU environment probe (zero GPU quota)

Two GPU sessions were lost to dependency failures (a P100 assignment, then a vLLM CUDA-variant
mismatch). Kaggle CPU notebooks don't consume GPU quota, so this establishes the environment
facts *for free* before any further GPU spend:

- installed `transformers` / `torch` / `accelerate` versions, and `torch.version.cuda`
- free disk in `/kaggle/working` (model weights are GB-scale)
- that the HF download path works, via a tokenizer-only load of each candidate
- the **measured** token length of a real per-label prompt, which is what the Step 1a
  throughput projection depends on

Nothing here needs a GPU. If a candidate's tokenizer won't download here, its weights won't
download in the GPU session either -- that is exactly what this is for.

In [ ]:
# Mount the private src/knee dataset. GIT_SHA is baked in at push time since
# Kaggle kernels have no git context.
import glob, os, shutil, sys, time

GIT_SHA = 'uncommitted-phase3-step1'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')
print('src/knee mounted from', SRC)

In [ ]:
import torch, transformers, shutil as _shutil
print('torch       ', torch.__version__)
print('torch.version.cuda', torch.version.cuda)
print('transformers', transformers.__version__)
try:
    import accelerate; print('accelerate  ', accelerate.__version__)
except ImportError:
    print('accelerate   NOT INSTALLED -- device_map="auto" (needed for the 8B across 2 GPUs) '
          'will fail; install it in the GPU notebook')

usage = _shutil.disk_usage('/kaggle/working')
print(f'\n/kaggle/working free: {usage.free / 1e9:.1f} GB of {usage.total / 1e9:.1f} GB')
print('(a 4B fp16 checkpoint is ~8GB, an 8B ~16GB -- HF caches to ~/.cache by default)')
home = _shutil.disk_usage(os.path.expanduser('~'))
print(f'home free: {home.free / 1e9:.1f} GB')

In [ ]:
import numpy as np
import pandas as pd

from knee.infer import LABEL_COLUMNS
from knee.reports import build_label_prompt, build_lexical_labels

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
gold_df = train_df[train_df['ACL'].notna()].reset_index(drop=True)
assert len(gold_df) == 58, f'expected 58 gold studies, got {len(gold_df)}'
assert gold_df['Report'].notna().all(), 'a gold study is missing report text'

reports = gold_df['Report'].tolist()
study_uids = gold_df['StudyInstanceUID'].tolist()
y_true = gold_df[LABEL_COLUMNS].to_numpy(dtype=float)
print(f'{len(gold_df)} gold studies loaded, all with report text')

In [ ]:
# Gated repos (Gemma) need an HF token whose account accepted the license.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN secret found')
except Exception as e:
    print(f'No HF_TOKEN Kaggle Secret ({type(e).__name__}) -- gated repos will fail below')

In [ ]:
# Tokenizer-only load per candidate: proves the download path and the chat
# template work, and measures real prompt lengths. Costs no GPU and ~1MB each.
from transformers import AutoTokenizer

CANDIDATES = [
    ('Qwen3-4B-Instruct', 'Qwen/Qwen3-4B-Instruct-2507'),
    ('Qwen3-8B',          'Qwen/Qwen3-8B'),
    ('Gemma-3-12B-IT',    'google/gemma-3-12b-it'),
]

# Longest real report in the corpus, not a median one -- the prompt-length
# number that matters for a memory/throughput budget is the worst case.
longest_report = max(train_df['Report'].dropna(), key=len)
median_report = sorted(train_df['Report'].dropna(), key=len)[len(train_df['Report'].dropna()) // 2]

for name, repo in CANDIDATES:
    print(f'=== {name} ({repo}) ===')
    try:
        tok = AutoTokenizer.from_pretrained(repo)
    except Exception as e:
        print(f'  TOKENIZER LOAD FAILED: {type(e).__name__}: {e}')
        continue

    # enable_thinking=False matters for Qwen3, whose template otherwise emits a
    # <think> block -- the first generated token would be the start of a
    # reasoning trace, not Yes/No. Gemma's template rejects the kwarg, so try
    # it and fall back rather than assuming either way.
    def render(prompt):
        msgs = [{'role': 'user', 'content': prompt}]
        try:
            return tok.apply_chat_template(msgs, tokenize=False,
                                           add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    supports_thinking_flag = True
    try:
        tok.apply_chat_template([{'role': 'user', 'content': 'x'}], tokenize=False,
                                add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        supports_thinking_flag = False
    print(f'  chat template accepts enable_thinking: {supports_thinking_flag}')

    for label_of_interest, report, tag in [
        ('ACL', median_report, 'median report'),
        ('ACL', longest_report, 'LONGEST report'),
    ]:
        n = len(tok(render(build_label_prompt(report, label_of_interest)))['input_ids'])
        print(f'  {tag}: {n} tokens/prompt -> x12 labels = {n * 12} tokens/report'
              f' -> x4407 reports = {n * 12 * 4407 / 1e6:.1f}M tokens for the full corpus')

    # single-token Yes/No is what the scorer reads; if a tokenizer splits them
    # the adapter has nothing to score and every answer would come back NaN
    single = []
    for word in ('Yes', 'No', ' Yes', ' No', 'yes', 'no'):
        ids = tok.encode(word, add_special_tokens=False)
        if len(ids) == 1:
            single.append(repr(word))
    print(f'  single-token yes/no forms available: {single if single else "NONE -- SCORER WOULD FAIL"}')

In [ ]:
print('Probe complete. Carry forward into Step 1a:')
print(' - the measured tokens/report above (throughput projection)')
print(' - which candidates tokenized at all (a failure here = a failure there)')
print(' - whether accelerate is present (needed for the 8B across both T4s)')